# 05 · Simüle Veri Seti Üretimi (FM Passive Radar)

Mevcut görsellerde (`data/train`) gözetleme kanalı referansla aynıydı (`cc_detector_fd(s, s)`), yani fiziksel bir
hedef yankısı yoktu. Bu notebook etiketleri **fiziksel olarak tanımlı** yeni bir veri seti üretir:

* **Aydınlatıcı:** ses kaydıyla FM modüle edilmiş kompleks taban bant sinyal (75 kHz sapma, 220.5 kHz örnekleme).
* **target:** gözetleme kanalı = gürültü + aydınlatıcının gecikmeli (bistatik menzil), Doppler kaydırmalı ve
  zayıflatılmış (SNR) kopyası.
* **no_target:** gözetleme kanalı = yalnızca gürültü.
* Her ses segmentinden **aynı aydınlatıcıyla** bir `target` ve bir `no_target` haritası üretilir; böylece sınıflar
  arasındaki tek sistematik fark hedef yankısıdır (ses içeriği değil).
* Range-Doppler haritası (pyAPRiL `cc_detector_fd` ile aynı algoritma) ham dB matrisi olarak `.npy` kaydedilir;
  hedefin menzil, Doppler ve SNR değerleri `metadata.csv` dosyasına yazılır.

Varsayımlar / sınırlılıklar: ideal (gürültüsüz) referans kanalı, doğrudan yol girişimi (direct-path) ve clutter yok,
tek hedef, hedef hızı entegrasyon süresince sabit.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "passive_radar").is_dir())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from passive_radar.config import CLASS_NAMES, ProjectConfig
from passive_radar.evaluation.plots import plot_rd_maps
from passive_radar.simulation.ambiguity import doppler_bin_count, doppler_step_hz
from passive_radar.simulation.dataset_builder import generate_dataset, list_audio_files, simulate_pair
from passive_radar.simulation.fm import load_audio_mono
from passive_radar.simulation.scenario import range_cells

config = ProjectConfig.default(PROJECT_ROOT)
simulation = config.simulation
AUDIO_DIR = config.paths.audio_dir          # change here if the recordings live elsewhere
OUTPUT_DIR = config.paths.simulated_dir     # data/simulated (existing data files are not touched)
REGENERATE = False                          # True: overwrite an existing simulated dataset
print(asdict(simulation))

## 1. Ses kayıtları ve radar geometrisi

In [ ]:
audio_files = list_audio_files(AUDIO_DIR)
print(f"{len(audio_files)} audio files in {AUDIO_DIR}")

audio, audio_rate = load_audio_mono(audio_files[0])
sample_rate = audio_rate * simulation.upsample_factor
num_samples = int(simulation.integration_time_s * sample_rate)
print(f"audio rate {audio_rate} Hz -> baseband {sample_rate / 1e3:.1f} kHz, N = {num_samples} samples")
print(f"range cell = {3e8 / sample_rate / 1e3:.2f} km, cells up to {simulation.max_range_km:.0f} km: "
      f"{range_cells(simulation.max_range_km, sample_rate)}")
print(f"Doppler step = {doppler_step_hz(sample_rate, num_samples):.2f} Hz, "
      f"+-{simulation.max_doppler_hz:.1f} Hz -> {2 * doppler_bin_count(sample_rate, num_samples, simulation.max_doppler_hz) + 1} bins")
print(f"coherent processing gain = {10 * np.log10(num_samples):.1f} dB "
      f"-> integrated target SNR {simulation.target_snr_db[0] + 10 * np.log10(num_samples):.0f} .. "
      f"{simulation.target_snr_db[1] + 10 * np.log10(num_samples):.0f} dB")

## 2. Örnek: aynı segmentten target / no_target haritası

In [ ]:
rng = np.random.default_rng(simulation.seed)
demo_config = replace(simulation, target_snr_db=(-30.0, -30.0))  # clearly visible for illustration
maps, target = simulate_pair(audio, audio_rate, start=audio_rate, config=demo_config, rng=rng)
print(target)
plot_rd_maps(
    [maps[CLASS_NAMES.index("target")], maps[CLASS_NAMES.index("no_target")]],
    ["target (circle = true position)", "no_target (same illuminator)"],
    simulation.max_range_km, simulation.max_doppler_hz,
    markers=[(target.bistatic_range_km, target.doppler_hz), None],
)
plt.show()

## 3. Veri setinin üretilmesi

In [ ]:
metadata_path = OUTPUT_DIR / "metadata.csv"
if metadata_path.exists() and not REGENERATE:
    print(f"Existing dataset found ({metadata_path}); set REGENERATE = True to rebuild it.")
else:
    samples = generate_dataset(audio_files, OUTPUT_DIR, simulation)
    print(f"generated {len(samples)} maps")

## 4. Üretilen veri setinin özeti

In [ ]:
metadata = pd.read_csv(metadata_path)
metadata["class"] = [CLASS_NAMES[i] for i in metadata["label"]]
print(metadata["class"].value_counts().to_string())
print("recordings (groups):", metadata["source_id"].nunique())
targets = metadata[metadata["label"] == CLASS_NAMES.index("target")]
targets[["target_range_km", "target_doppler_hz", "target_snr_db"]].describe().round(2)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(14, 3))
for axis, column in zip(axes, ["target_range_km", "target_doppler_hz", "target_snr_db"]):
    axis.hist(targets[column], bins=20)
    axis.set_title(column)
plt.tight_layout()
plt.show()